In [15]:
numbers = [10, 20, 30]
try:
    print(hash(numbers)) #TypeError: unhashable type: 'list'
except TypeError as e:
    print(f"Error: {e}")

Error: unhashable type: 'list'


In [1]:
import os
import pyspark
from delta import configure_spark_with_delta_pip

# Ensure Windows points to your local Hadoop binaries
os.environ["HADOOP_HOME"] = "C:\\hadoop"

# Configure SparkSession with Delta Lake extensions
builder = (
    pyspark.sql.SparkSession.builder.appName("LocalDeltaLakehouse")
    .master("local[*]")  # Uses all available CPU cores on your Windows PC
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.driver.memory", "4g")  # Adjust based on your PC's RAM
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()

print(f"Spark Version: {spark.version}")
print("Delta Lake initialized successfully!")


Spark Version: 3.5.5
Delta Lake initialized successfully!


In [2]:
import os
os.environ["HADOOP_HOME"] = "C:\\hadoop"
os.environ["PATH"] += os.pathsep + "C:\\hadoop\\bin"

In [3]:
import importlib.metadata
import pyspark

print(f"PySpark: {pyspark.__version__}")
print(f"Delta: {importlib.metadata.version('delta-spark')}")

PySpark: 3.5.5
Delta: 3.0.0


In [4]:
import os
import sys
import pyspark
from delta import configure_spark_with_delta_pip

# 1. Force Windows to find winutils.exe and hadoop.dll
os.environ["HADOOP_HOME"] = "C:\\hadoop"
os.environ["PATH"] += os.pathsep + "C:\\hadoop\\bin"

# 2. Point PySpark explicitly to the active Conda Python executable
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

builder = (
    pyspark.sql.SparkSession.builder.appName("LocalDeltaFix")
    .master("local[*]")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.driver.host", "127.0.0.1")  # Prevents Windows firewall/network socket blocks
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()
print("Spark & Delta initialized cleanly!")

Spark & Delta initialized cleanly!


In [5]:
import os

for fname in ["winutils.exe", "hadoop.dll"]:
    fpath = os.path.join("C:\\hadoop\\bin", fname)
    if os.path.exists(fpath):
        print(f"{fname}: Found ({os.path.getsize(fpath)} bytes)")
    else:
        print(f"{fname}: MISSING!")

winutils.exe: Found (112640 bytes)
hadoop.dll: Found (84992 bytes)


In [6]:
hadoop_version = spark.sparkContext._jvm.org.apache.hadoop.util.VersionInfo.getVersion()
print(f"PySpark Hadoop Version: {hadoop_version}")

PySpark Hadoop Version: 3.3.4


In [9]:
import os
import sys
import pyspark
from delta import configure_spark_with_delta_pip

# 1. Stop any lingering Spark sessions first
try:
    spark.stop()
except Exception:
    pass

# 2. Add Conda environment DLL directories so spawned Python workers don't crash on startup
conda_env_dir = os.path.dirname(sys.executable)
conda_lib_bin = os.path.join(conda_env_dir, "Library", "bin")
conda_scripts = os.path.join(conda_env_dir, "Scripts")

os.environ["PATH"] = f"{conda_env_dir};{conda_lib_bin};{conda_scripts};C:\\hadoop\\bin;" + os.environ["PATH"]
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["HADOOP_HOME"] = "C:\\hadoop"

java_opts = (
    "-Djava.library.path=C:\\hadoop\\bin "
    "--add-opens=java.base/java.lang=ALL-UNNAMED "
    "--add-opens=java.base/java.lang.invoke=ALL-UNNAMED "
    "--add-opens=java.base/java.lang.reflect=ALL-UNNAMED "
    "--add-opens=java.base/java.io=ALL-UNNAMED "
    "--add-opens=java.base/java.net=ALL-UNNAMED "
    "--add-opens=java.base/java.nio=ALL-UNNAMED "
    "--add-opens=java.base/java.util=ALL-UNNAMED "
    "--add-opens=java.base/java.util.concurrent=ALL-UNNAMED "
    "--add-opens=java.base/java.util.concurrent.atomic=ALL-UNNAMED "
    "--add-opens=java.base/sun.nio.ch=ALL-UNNAMED "
    "--add-opens=java.base/sun.nio.cs=ALL-UNNAMED "
    "--add-opens=java.base/sun.security.action=ALL-UNNAMED "
    "--add-opens=java.base/sun.util.calendar=ALL-UNNAMED "
    "--add-opens=java.security.jgss/sun.security.krb5=ALL-UNNAMED "
    "-Djdk.security.auth.subject.useTL=true"
)

# 3. Force single-worker execution (local[1]) to prevent Windows socket exhaustion
builder = (
    pyspark.sql.SparkSession.builder.appName("LocalDeltaLakehouse")
    .master("local[1]")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.driver.extraJavaOptions", java_opts)
    .config("spark.executor.extraJavaOptions", java_opts)
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.python.worker.reuse", "true")
    .config("spark.sql.shuffle.partitions", "1")
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()

# 4. Test writing and reading the Delta table
delta_path = "file:///C:/temp/delta_vault_test"
df = spark.createDataFrame([("CUST_001", "2026-11-11", "CORE_BANKING")], ["customer_hk", "load_date", "record_source"])
df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(delta_path)

print("Delta table written successfully!")
spark.read.format("delta").load(delta_path).show()

Delta table written successfully!
+-----------+----------+-------------+
|customer_hk| load_date|record_source|
+-----------+----------+-------------+
|   CUST_001|2026-11-11| CORE_BANKING|
+-----------+----------+-------------+



In [11]:
from delta.tables import DeltaTable

# Define a local folder path for your Delta table
delta_path = "C:/users/Hethi/Downloads/spark-NA/work/delta_vault_test"

# 1. Create sample Data Vault Hub data (Version 0)
data_v0 = [
    ("CUST_001", "2026-11-11", "CORE_BANKING"),
    ("CUST_002", "2026-11-11", "CARDS_SYSTEM")
]
columns = ["customer_hk", "load_date", "record_source"]

df = spark.createDataFrame(data_v0, columns)
df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(delta_path)

print("--- Version 0 (Initial Load) ---")
spark.read.format("delta").load(delta_path).show()

# 2. Append a new record (Version 1)
data_v1 = [("CUST_003", "2026-11-11", "ERP_SYSTEM")]
spark.createDataFrame(data_v1, columns).write.format("delta").mode("append").save(delta_path)

print("--- Version 1 (After Append) ---")
spark.read.format("delta").load(delta_path).show()

# 3. Test Delta Time Travel (Read Version 0 again)
print("--- Time Travel Check (Querying Version 0) ---")
spark.read.format("delta").option("versionAsOf", 0).load(delta_path).show()


--- Version 0 (Initial Load) ---
+-----------+----------+-------------+
|customer_hk| load_date|record_source|
+-----------+----------+-------------+
|   CUST_001|2026-11-11| CORE_BANKING|
|   CUST_002|2026-11-11| CARDS_SYSTEM|
+-----------+----------+-------------+

--- Version 1 (After Append) ---
+-----------+----------+-------------+
|customer_hk| load_date|record_source|
+-----------+----------+-------------+
|   CUST_001|2026-11-11| CORE_BANKING|
|   CUST_002|2026-11-11| CARDS_SYSTEM|
|   CUST_003|2026-11-11|   ERP_SYSTEM|
+-----------+----------+-------------+

--- Time Travel Check (Querying Version 0) ---
+-----------+----------+-------------+
|customer_hk| load_date|record_source|
+-----------+----------+-------------+
|   CUST_001|2026-11-11| CORE_BANKING|
|   CUST_002|2026-11-11| CARDS_SYSTEM|
+-----------+----------+-------------+



In [12]:
# Runs 100% in the JVM with zero Python worker sockets
delta_path = "C:/users/Hethi/Downloads/spark-NA/work/delta_vault_test"
df_sql = spark.sql("""
    SELECT 
        'CUST_001' AS customer_hk, 
        DATE('2026-11-11') AS load_date, 
        'CORE_BANKING' AS record_source
""")
#Add .option("overwriteSchema", "true") to your .write command so Delta Lake replaces the old StringType table definition with your new DateType definition:
df_sql.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(delta_path)
spark.read.format("delta").load(delta_path).show()


+-----------+----------+-------------+
|customer_hk| load_date|record_source|
+-----------+----------+-------------+
|   CUST_001|2026-11-11| CORE_BANKING|
+-----------+----------+-------------+



In [13]:
# Define a local folder path for your Delta table
delta_path = "C:/users/Hethi/Downloads/spark-NA/work/delta_vault_test"
# 1. List all available catalogs (similar to Unity Catalog top-level view)
spark.sql("SHOW CATALOGS").show()
# 2. List all schemas / databases
spark.sql("SHOW SCHEMAS").show()
# 3. Register your local Delta folder as a named table in a schema
spark.sql("CREATE SCHEMA IF NOT EXISTS dv2_silver")
spark.sql("DROP TABLE IF EXISTS dv2_silver.hub_customer")
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS dv2_silver.hub_customer 
    USING DELTA 
    LOCATION '{delta_path}'
""")
# 4. List all tables inside your schema
spark.sql("SHOW TABLES IN dv2_silver").show()
# 5. View column names, data types, and comments (like Databricks Schema tab)
spark.sql("DESCRIBE TABLE EXTENDED dv2_silver.hub_customer").show(truncate=False)

+-------------+
|      catalog|
+-------------+
|spark_catalog|
+-------------+

+---------+
|namespace|
+---------+
|  default|
+---------+

+----------+------------+-----------+
| namespace|   tableName|isTemporary|
+----------+------------+-----------+
|dv2_silver|hub_customer|      false|
+----------+------------+-----------+

+----------------------------+-------------------------------------------------------------+-------+
|col_name                    |data_type                                                    |comment|
+----------------------------+-------------------------------------------------------------+-------+
|customer_hk                 |string                                                       |NULL   |
|load_date                   |date                                                         |NULL   |
|record_source               |string                                                       |NULL   |
|                            |                               

In [14]:
def show_catalog_explorer():
    print("🗄️  SPARK CATALOG EXPLORER")
    print("=" * 55)
    
    # Iterate through catalogs
    catalogs = [row.catalog for row in spark.sql("SHOW CATALOGS").collect()]
    for cat in catalogs:
        print(f"📁 Catalog: {cat}")
        spark.catalog.setCurrentCatalog(cat)
        
        # Iterate through schemas (databases)
        databases = spark.catalog.listDatabases()
        for db in databases:
            print(f"  └── 📂 Schema: {db.name}")
            
            # Iterate through tables in each schema
            tables = spark.catalog.listTables(db.name)
            if not tables:
                print(f"        └── (No tables registered)")
            for tbl in tables:
                print(f"        └── 📄 Table: {tbl.name} [{tbl.tableType}]")
                
                # List columns and data types
                #columns = spark.catalog.listColumns(tbl.name, db.name)
                columns = spark.catalog.listColumns(f"{db.name}.{tbl.name}")
                for col in columns:
                    pk_flag = "🔑" if col.isPartition else "  "
                    print(f"              {pk_flag} {col.name}: {col.dataType} (nullable={col.nullable})")
    print("=" * 55)

# Run the explorer
show_catalog_explorer()

🗄️  SPARK CATALOG EXPLORER
📁 Catalog: spark_catalog
  └── 📂 Schema: default
        └── (No tables registered)
  └── 📂 Schema: dv2_silver
        └── 📄 Table: hub_customer [EXTERNAL]
                 customer_hk: string (nullable=True)
                 load_date: date (nullable=True)
                 record_source: string (nullable=True)
